In [ ]:
import rawpy
import matplotlib.pyplot as plt
import cv2
import numpy as np
from RawDeveloping.RawPostprocess import RawPostprocess
import torch

In [ ]:
## Get Raw Image

In [ ]:
path = "/Users/ryanmueller/Pictures/India_2024/DSC01182.ARW"

In [ ]:
rawimg = rawpy.imread(path)

In [ ]:
_raw = rawimg.postprocess(use_camera_wb=True, gamma=(1, 1))


In [ ]:
raw = _raw[:, :-59]/255
raw.shape

In [ ]:

raw = cv2.resize(raw, (224, 224), interpolation=cv2.INTER_AREA)

In [ ]:
raw_tensor = torch.tensor(raw).permute(2, 0, 1).unsqueeze(0)
raw_tensor.shape

In [ ]:
postprocess = RawPostprocess()

In [ ]:
raw_tensor_procesed = postprocess(raw_tensor)

In [ ]:
plt.imshow(raw_tensor_procesed[0].permute(1, 2, 0))

In [ ]:
import os
import torch
import torch.nn as nn
from os.path import expanduser  # pylint: disable=import-outside-toplevel
from urllib.request import urlretrieve  # pylint: disable=import-outside-toplevel
def get_aesthetic_model(clip_model="vit_l_14"):
    """load the aethetic model"""
    home = expanduser("~")
    cache_folder = home + "/.cache/emb_reader"
    path_to_model = cache_folder + "/sa_0_4_"+clip_model+"_linear.pth"
    if not os.path.exists(path_to_model):
        os.makedirs(cache_folder, exist_ok=True)
        url_model = (
            "https://github.com/LAION-AI/aesthetic-predictor/blob/main/sa_0_4_"+clip_model+"_linear.pth?raw=true"
        )
        urlretrieve(url_model, path_to_model)
    if clip_model == "vit_l_14":
        m = nn.Linear(768, 1)
    elif clip_model == "vit_b_32":
        m = nn.Linear(512, 1)
    else:
        raise ValueError()
    s = torch.load(path_to_model)
    m.load_state_dict(s)
    m.eval()
    return m

amodel= get_aesthetic_model(clip_model="vit_l_14")
amodel.eval()

import torch
from PIL import Image
import open_clip
model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')

In [ ]:
def make_image():
    img = postprocess(raw_tensor.float())
    img = (img * 4) - 2
    return img
    

In [ ]:
device = 'cpu'

raw = _raw[:, :-59]/255

raw = cv2.resize(raw, (224, 224), interpolation=cv2.INTER_AREA)
raw_tensor = torch.tensor(raw).permute(2, 0, 1).unsqueeze(0)
postprocess = RawPostprocess(learnable=True).to(device)
raw_tensor = raw_tensor.float().to(device)
model = model.to(device)
amodel = amodel.to(device)
opt = torch.optim.Adam(postprocess.parameters(), lr=1e-2)

In [ ]:
# postprocess.r.requires_grad = False
# postprocess.g.requires_grad = False
# postprocess.b.requires_grad = False
postprocess.gamma.requires_grad = False

In [ ]:
raw = cv2.resize(raw, (224, 224), interpolation=cv2.INTER_AREA)
raw_tensor = torch.tensor(raw).permute(2, 0, 1).unsqueeze(0)
raw_tensor = raw_tensor.float().to(device)

In [ ]:
import torchvision.transforms.functional as TF

for i in range(60):
    img = postprocess(raw_tensor)

    clip_mean = [0.48145466, 0.4578275, 0.40821073]
    clip_std = [0.26862954, 0.26130258, 0.27577711]
    clip_mean = [0.45, 0.45, 0.45]
    clip_std = [0.27, 0.27, 0.27]
    clip_std = [0.5, 0.5, 0.5]

    img_normalized = TF.normalize(img, mean=clip_mean, std=clip_std)

    image_features = model.encode_image(img_normalized)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    prediction = amodel(image_features)
    loss = -prediction.mean() 
    
    opt.zero_grad()
    loss.backward()
    opt.step()
    print(loss)
print(prediction)

In [ ]:
raw = _raw[:, :-59]/255

raw = cv2.resize(raw, (raw.shape[1]//14, raw.shape[0]//14), interpolation=cv2.INTER_AREA)
raw_tensor = torch.tensor(raw).permute(2, 0, 1).unsqueeze(0)
postprocess = postprocess.to('cpu')

postprocess2 = RawPostprocess(learnable=False)

with torch.no_grad():
    raw_tensor_procesed = postprocess(raw_tensor)
    raw_tensor_default_procesed = postprocess2(raw_tensor)

In [ ]:
plt.imshow(raw_tensor_procesed[0].permute(1, 2, 0))

In [ ]:
plt.imshow(raw_tensor_default_procesed[0].permute(1, 2, 0))